# Gold Maintenance KPI Marts

This notebook builds Gold operational marts from the Silver maintenance dataset.

It produces three datasets:
- `daily_operations`: event volume, AOG load, open backlog, and severity mix by day and location
- `component_reliability`: component-level daily reliability and maintenance pressure indicators
- `fleet_status_snapshot`: latest known maintenance status per aircraft tail number

In Databricks, the notebook writes Delta datasets to the Gold container. In local mode, it computes previews from the latest raw capture file without writing Delta.

In [1]:
import os

storage_account = os.getenv("JETOPS_STORAGE_ACCOUNT", "stherbalifedev001")
raw_container = os.getenv("JETOPS_RAW_CONTAINER", "raw")
silver_container = os.getenv("JETOPS_SILVER_CONTAINER", "silver")
gold_container = os.getenv("JETOPS_GOLD_CONTAINER", "gold")
secret_scope = os.getenv("JETOPS_SECRET_SCOPE", "herbalife-storage")
raw_secret_key = os.getenv("JETOPS_SECRET_KEY", "raw-sas-token")
silver_secret_key = os.getenv("JETOPS_SILVER_SECRET_KEY", raw_secret_key)
gold_secret_key = os.getenv("JETOPS_GOLD_SECRET_KEY", raw_secret_key)
storage_account_key_secret = os.getenv("JETOPS_STORAGE_ACCOUNT_KEY_SECRET", "storage-account-key")
storage_auth_mode = os.getenv("JETOPS_STORAGE_AUTH_MODE", "account_key")
eventhub_namespace = os.getenv("JETOPS_EVENTHUB_NAMESPACE", "evh-herbalife-dev")
eventhub_name = os.getenv("JETOPS_EVENTHUB_NAME", "jetops-maintenance-events-dev")
capture_root = os.getenv("JETOPS_CAPTURE_ROOT", "jetops-maintenance")
resource_group = os.getenv("JETOPS_RESOURCE_GROUP", "rg-herbalife-dev-core")
silver_dataset = os.getenv("JETOPS_SILVER_DATASET", "jetops/maintenance_events")
gold_dataset_root = os.getenv("JETOPS_GOLD_DATASET_ROOT", "jetops/maintenance_kpis")
daily_operations_dataset = os.getenv("JETOPS_GOLD_DAILY_DATASET", f"{gold_dataset_root}/daily_operations")
component_reliability_dataset = os.getenv("JETOPS_GOLD_COMPONENT_DATASET", f"{gold_dataset_root}/component_reliability")
fleet_status_dataset = os.getenv("JETOPS_GOLD_FLEET_DATASET", f"{gold_dataset_root}/fleet_status_snapshot")
write_mode = os.getenv("JETOPS_GOLD_WRITE_MODE", "overwrite")
az_cli = os.getenv("AZURE_CLI_PATH", r"C:\Program Files\Microsoft SDKs\Azure\CLI2\wbin\az.cmd")

silver_delta_path = f"wasbs://{silver_container}@{storage_account}.blob.core.windows.net/{silver_dataset}"
daily_operations_path = f"wasbs://{gold_container}@{storage_account}.blob.core.windows.net/{daily_operations_dataset}"
component_reliability_path = f"wasbs://{gold_container}@{storage_account}.blob.core.windows.net/{component_reliability_dataset}"
fleet_status_path = f"wasbs://{gold_container}@{storage_account}.blob.core.windows.net/{fleet_status_dataset}"
raw_capture_prefix = f"{capture_root}/{eventhub_namespace}/{eventhub_name}"

is_databricks = "dbutils" in globals() and "spark" in globals()
print(f"Execution mode: {'databricks' if is_databricks else 'local'}")
print(f"Silver Delta path: {silver_delta_path}")
print(f"Gold dataset root: wasbs://{gold_container}@{storage_account}.blob.core.windows.net/{gold_dataset_root}")
print(f"Write mode: {write_mode}")
print(f"Databricks storage auth mode: {storage_auth_mode}")

if is_databricks:
    if storage_auth_mode == "sas":
        silver_sas_token = dbutils.secrets.get(scope=secret_scope, key=silver_secret_key)
        gold_sas_token = dbutils.secrets.get(scope=secret_scope, key=gold_secret_key)
        spark.conf.set(
            f"fs.azure.sas.{silver_container}.{storage_account}.blob.core.windows.net",
            silver_sas_token,
        )
        spark.conf.set(
            f"fs.azure.sas.{gold_container}.{storage_account}.blob.core.windows.net",
            gold_sas_token,
        )
    else:
        storage_account_key = dbutils.secrets.get(scope=secret_scope, key=storage_account_key_secret)
        spark.conf.set(
            f"fs.azure.account.key.{storage_account}.blob.core.windows.net",
            storage_account_key,
        )
else:
    print("Local mode will compute Gold previews from the latest raw capture file.")

Execution mode: local
Silver Delta path: wasbs://silver@stherbalifedev001.blob.core.windows.net/jetops/maintenance_events
Gold dataset root: wasbs://gold@stherbalifedev001.blob.core.windows.net/jetops/maintenance_kpis
Write mode: overwrite
Databricks storage auth mode: account_key
Local mode will compute Gold previews from the latest raw capture file.


In [ ]:
import json
import subprocess
import tempfile
from datetime import UTC, date, datetime
from pathlib import Path

if is_databricks:
    from pyspark.sql.functions import (
        avg,
        col,
        count,
        countDistinct,
        current_date,
        current_timestamp,
        datediff,
        lit,
        max as spark_max,
        row_number,
        sum as spark_sum,
        to_date,
        to_timestamp,
        unix_timestamp,
        when,
    )
    from pyspark.sql.window import Window

    silver_df = (
        spark.read.format("delta").load(silver_delta_path)
        .withColumn("event_timestamp", to_timestamp(col("event_timestamp")))
        .withColumn("inspection_date", to_date(col("inspection_date")))
        .withColumn("silver_loaded_at", to_timestamp(col("silver_loaded_at")))
        .withColumn("event_date", to_date(col("event_timestamp")))
    )

    open_statuses = ["Open", "In-Work", "Awaiting Parts", "AOG", "Return To Service Review"]
    silver_enriched_df = (
        silver_df
        .withColumn("is_open", when(col("status").isin(open_statuses), lit(1)).otherwise(lit(0)))
        .withColumn("is_aog", when(col("status") == lit("AOG"), lit(1)).otherwise(lit(0)))
        .withColumn("is_critical", when(col("severity") == lit("Critical"), lit(1)).otherwise(lit(0)))
        .withColumn("is_high_or_critical", when(col("severity").isin("High", "Critical"), lit(1)).otherwise(lit(0)))
        .withColumn("is_unscheduled", when(col("maintenance_type").isin("Unscheduled Repair", "AOG Recovery", "Diagnostics"), lit(1)).otherwise(lit(0)))
        .withColumn("requires_parts_flag", when(col("requires_parts") == lit(1), lit(1)).otherwise(lit(0)))
        .withColumn("repeat_issue_indicator", when(col("repeat_issue_flag") == lit(1), lit(1)).otherwise(lit(0)))
        .withColumn("inspection_age_hours", (unix_timestamp(col("event_timestamp")) - unix_timestamp(col("inspection_date"))) / lit(3600.0))
        .withColumn("gold_loaded_at", current_timestamp())
        .withColumn("snapshot_date", current_date())
    )

    daily_operations_df = (
        silver_enriched_df
        .groupBy("event_date", "airport_code", "hangar")
        .agg(
            count(lit(1)).alias("total_events"),
            countDistinct("tail_number").alias("affected_aircraft"),
            countDistinct("component").alias("distinct_components"),
            spark_sum("is_open").alias("open_events"),
            spark_sum("is_aog").alias("aog_events"),
            spark_sum("is_critical").alias("critical_events"),
            spark_sum("is_high_or_critical").alias("high_or_critical_events"),
            spark_sum("requires_parts_flag").alias("parts_required_events"),
            spark_sum("repeat_issue_indicator").alias("repeat_issue_events"),
            avg("part_hours").alias("avg_part_hours"),
            avg("estimated_downtime_hours").alias("avg_estimated_downtime_hours"),
            avg("labor_hours_estimate").alias("avg_labor_hours_estimate"),
            avg("inspection_age_hours").alias("avg_inspection_age_hours"),
            spark_max("event_timestamp").alias("last_event_timestamp")
        )
        .withColumn("gold_loaded_at", current_timestamp())
    )

    component_reliability_df = (
        silver_enriched_df
        .groupBy("event_date", "component")
        .agg(
            count(lit(1)).alias("total_events"),
            countDistinct("tail_number").alias("affected_aircraft"),
            spark_sum("is_aog").alias("aog_events"),
            spark_sum("is_critical").alias("critical_events"),
            spark_sum("is_unscheduled").alias("unscheduled_events"),
            spark_sum("requires_parts_flag").alias("parts_required_events"),
            spark_sum("repeat_issue_indicator").alias("repeat_issue_events"),
            avg("part_hours").alias("avg_part_hours"),
            avg("estimated_downtime_hours").alias("avg_estimated_downtime_hours"),
            avg("labor_hours_estimate").alias("avg_labor_hours_estimate"),
            avg("inspection_age_hours").alias("avg_inspection_age_hours"),
            spark_max("event_timestamp").alias("last_event_timestamp")
        )
        .withColumn("gold_loaded_at", current_timestamp())
    )

    fleet_window = Window.partitionBy("tail_number").orderBy(
        col("event_timestamp").desc_nulls_last(),
        col("silver_loaded_at").desc_nulls_last(),
        col("sequence_number").desc_nulls_last()
    )

    fleet_status_df = (
        silver_enriched_df
        .withColumn("fleet_row_rank", row_number().over(fleet_window))
        .filter(col("fleet_row_rank") == 1)
        .drop("fleet_row_rank")
        .select(
            "snapshot_date",
            "tail_number",
            "aircraft_model",
            "airport_code",
            "hangar",
            col("event_timestamp").alias("latest_event_timestamp"),
            col("status").alias("current_status"),
            col("severity").alias("current_severity"),
            col("component").alias("current_component"),
            col("maintenance_type").alias("current_maintenance_type"),
            col("fault_code").alias("current_fault_code"),
            col("technician_id").alias("current_technician_id"),
            col("priority").alias("current_priority"),
            col("dispatch_impact").alias("current_dispatch_impact"),
            col("operator_name").alias("current_operator_name"),
            col("route_segment").alias("current_route_segment"),
            col("maintenance_station").alias("current_maintenance_station"),
            col("part_order_status").alias("current_part_order_status"),
            col("estimated_downtime_hours").alias("current_estimated_downtime_hours"),
            col("labor_hours_estimate").alias("current_labor_hours_estimate"),
            col("is_open").alias("open_issue_flag"),
            col("is_aog").alias("aog_flag"),
            col("requires_parts_flag").alias("requires_parts_flag"),
            col("repeat_issue_indicator").alias("repeat_issue_flag"),
            datediff(current_date(), col("event_date")).alias("days_since_latest_event"),
            "gold_loaded_at"
        )
    )

    daily_writer = (
        daily_operations_df.write
        .format("delta")
        .mode(write_mode)
        .option("overwriteSchema", "true")
    )
    component_writer = (
        component_reliability_df.write
        .format("delta")
        .mode(write_mode)
        .option("overwriteSchema", "true")
    )
    fleet_writer = (
        fleet_status_df.write
        .format("delta")
        .mode(write_mode)
        .option("overwriteSchema", "true")
    )
    if write_mode != "overwrite":
        daily_writer = daily_writer.option("mergeSchema", "true")
        component_writer = component_writer.option("mergeSchema", "true")
        fleet_writer = fleet_writer.option("mergeSchema", "true")

    (
        daily_writer
        .partitionBy("event_date")
        .save(daily_operations_path)
    )
    (
        component_writer
        .partitionBy("event_date")
        .save(component_reliability_path)
    )
    (
        fleet_writer
        .partitionBy("snapshot_date")
        .save(fleet_status_path)
    )

    print(f"Wrote daily operations mart to {daily_operations_path}")
    print(f"Wrote component reliability mart to {component_reliability_path}")
    print(f"Wrote fleet status snapshot mart to {fleet_status_path}")
    print(f"Daily operations rows: {daily_operations_df.count()}")
    print(f"Component reliability rows: {component_reliability_df.count()}")
    print(f"Fleet status snapshot rows: {fleet_status_df.count()}")
else:
    try:
        from fastavro import reader
    except ImportError as exc:
        raise ImportError(
            "Local mode requires fastavro in the notebook kernel. Install it before running this cell."
        ) from exc

    account_key = subprocess.check_output(
        [
            az_cli,
            "storage",
            "account",
            "keys",
            "list",
            "--resource-group",
            resource_group,
            "--account-name",
            storage_account,
            "--query",
            "[0].value",
            "-o",
            "tsv",
        ],
        text=True,
    ).strip()

    file_list = subprocess.check_output(
        [
            az_cli,
            "storage",
            "fs",
            "file",
            "list",
            "--account-name",
            storage_account,
            "--account-key",
            account_key,
            "--file-system",
            raw_container,
            "--path",
            raw_capture_prefix,
            "--exclude-dir",
            "-o",
            "json",
        ],
        text=True,
    )
    files = json.loads(file_list)
    avro_files = sorted(file_info["name"] for file_info in files if file_info["name"].endswith(".avro"))
    if not avro_files:
        raise FileNotFoundError(f"No Avro capture files found under {raw_capture_prefix}")

    latest_file = avro_files[-1]
    with tempfile.TemporaryDirectory() as temp_dir:
        local_file = Path(temp_dir) / Path(latest_file).name
        subprocess.run(
            [
                az_cli,
                "storage",
                "fs",
                "file",
                "download",
                "--account-name",
                storage_account,
                "--account-key",
                account_key,
                "--file-system",
                raw_container,
                "--path",
                latest_file,
                "--destination",
                str(local_file),
                "--overwrite",
                "true",
            ],
            check=True,
            capture_output=True,
            text=True,
        )

        with local_file.open("rb") as handle:
            records = list(reader(handle))

    def _parse_event_timestamp(value):
        if not value:
            return None
        return datetime.fromisoformat(value.replace("Z", "+00:00"))

    latest_by_event_id = {}
    for record in records:
        body_json = record["Body"].decode("utf-8")
        payload = json.loads(body_json)
        event_id = (payload.get("event_id") or "").strip()
        event_timestamp = _parse_event_timestamp(payload.get("event_time_utc"))
        tail_number = (payload.get("tail_number") or "").strip().upper()
        component = (payload.get("component") or "").strip().title()
        if not event_id or not event_timestamp or not tail_number or not component:
            continue

        silver_row = {
            **payload,
            "event_id": event_id,
            "tail_number": tail_number,
            "airport_code": (payload.get("airport_code") or "").strip().upper(),
            "hangar": (payload.get("hangar") or "").strip().upper(),
            "component": component,
            "maintenance_type": (payload.get("maintenance_type") or "").strip().title(),
            "status": (payload.get("status") or "").strip().title(),
            "severity": (payload.get("severity") or "").strip().title(),
            "silver_loaded_at": datetime.now(UTC),
            "event_timestamp": event_timestamp,
            "event_date": event_timestamp.date(),
            "inspection_date": date.fromisoformat(payload["inspection_date"]) if payload.get("inspection_date") else None,
            "sequence_number": record.get("SequenceNumber"),
        }

        current = latest_by_event_id.get(event_id)
        if current is None or silver_row["event_timestamp"] > current["event_timestamp"] or (
            silver_row["event_timestamp"] == current["event_timestamp"] and (silver_row["sequence_number"] or -1) > (current["sequence_number"] or -1)
        ):
            latest_by_event_id[event_id] = silver_row

    silver_rows = list(latest_by_event_id.values())
    open_statuses = {"Open", "In-Work", "Awaiting Parts", "AOG", "Return To Service Review"}
    now_utc = datetime.now(UTC)

    daily_operations_acc = {}
    component_reliability_acc = {}
    fleet_status_acc = {}

    for row in silver_rows:
        event_date_value = row["event_date"].isoformat()
        airport_code = row.get("airport_code") or "UNKNOWN"
        hangar = row.get("hangar") or "UNKNOWN"
        component_name = row.get("component") or "Unknown"
        is_open = 1 if row.get("status") in open_statuses else 0
        is_aog = 1 if row.get("status") == "AOG" else 0
        is_critical = 1 if row.get("severity") == "Critical" else 0
        is_high_or_critical = 1 if row.get("severity") in {"High", "Critical"} else 0
        is_unscheduled = 1 if row.get("maintenance_type") in {"Unscheduled Repair", "AOG Recovery", "Diagnostics"} else 0
        requires_parts = 1 if row.get("requires_parts") == 1 else 0
        repeat_issue_flag = 1 if row.get("repeat_issue_flag") == 1 else 0
        inspection_date_value = row.get("inspection_date")
        inspection_age_hours = None
        if inspection_date_value is not None:
            inspection_age_hours = (row["event_timestamp"] - datetime.combine(inspection_date_value, datetime.min.time(), tzinfo=UTC)).total_seconds() / 3600.0

        daily_key = (event_date_value, airport_code, hangar)
        if daily_key not in daily_operations_acc:
            daily_operations_acc[daily_key] = {
                "event_date": event_date_value,
                "airport_code": airport_code,
                "hangar": hangar,
                "total_events": 0,
                "tail_numbers": set(),
                "components": set(),
                "open_events": 0,
                "aog_events": 0,
                "critical_events": 0,
                "high_or_critical_events": 0,
                "parts_required_events": 0,
                "repeat_issue_events": 0,
                "part_hours_sum": 0.0,
                "part_hours_count": 0,
                "estimated_downtime_sum": 0.0,
                "estimated_downtime_count": 0,
                "labor_hours_sum": 0.0,
                "labor_hours_count": 0,
                "inspection_age_sum": 0.0,
                "inspection_age_count": 0,
                "last_event_timestamp": row["event_timestamp"],
            }
        daily_bucket = daily_operations_acc[daily_key]
        daily_bucket["total_events"] += 1
        daily_bucket["tail_numbers"].add(row["tail_number"])
        daily_bucket["components"].add(component_name)
        daily_bucket["open_events"] += is_open
        daily_bucket["aog_events"] += is_aog
        daily_bucket["critical_events"] += is_critical
        daily_bucket["high_or_critical_events"] += is_high_or_critical
        daily_bucket["parts_required_events"] += requires_parts
        daily_bucket["repeat_issue_events"] += repeat_issue_flag
        if row.get("part_hours") is not None:
            daily_bucket["part_hours_sum"] += float(row["part_hours"])
            daily_bucket["part_hours_count"] += 1
        if row.get("estimated_downtime_hours") is not None:
            daily_bucket["estimated_downtime_sum"] += float(row["estimated_downtime_hours"])
            daily_bucket["estimated_downtime_count"] += 1
        if row.get("labor_hours_estimate") is not None:
            daily_bucket["labor_hours_sum"] += float(row["labor_hours_estimate"])
            daily_bucket["labor_hours_count"] += 1
        if inspection_age_hours is not None:
            daily_bucket["inspection_age_sum"] += inspection_age_hours
            daily_bucket["inspection_age_count"] += 1
        if row["event_timestamp"] > daily_bucket["last_event_timestamp"]:
            daily_bucket["last_event_timestamp"] = row["event_timestamp"]

        component_key = (event_date_value, component_name)
        if component_key not in component_reliability_acc:
            component_reliability_acc[component_key] = {
                "event_date": event_date_value,
                "component": component_name,
                "total_events": 0,
                "tail_numbers": set(),
                "aog_events": 0,
                "critical_events": 0,
                "unscheduled_events": 0,
                "parts_required_events": 0,
                "repeat_issue_events": 0,
                "part_hours_sum": 0.0,
                "part_hours_count": 0,
                "estimated_downtime_sum": 0.0,
                "estimated_downtime_count": 0,
                "labor_hours_sum": 0.0,
                "labor_hours_count": 0,
                "inspection_age_sum": 0.0,
                "inspection_age_count": 0,
                "last_event_timestamp": row["event_timestamp"],
            }
        component_bucket = component_reliability_acc[component_key]
        component_bucket["total_events"] += 1
        component_bucket["tail_numbers"].add(row["tail_number"])
        component_bucket["aog_events"] += is_aog
        component_bucket["critical_events"] += is_critical
        component_bucket["unscheduled_events"] += is_unscheduled
        component_bucket["parts_required_events"] += requires_parts
        component_bucket["repeat_issue_events"] += repeat_issue_flag
        if row.get("part_hours") is not None:
            component_bucket["part_hours_sum"] += float(row["part_hours"])
            component_bucket["part_hours_count"] += 1
        if row.get("estimated_downtime_hours") is not None:
            component_bucket["estimated_downtime_sum"] += float(row["estimated_downtime_hours"])
            component_bucket["estimated_downtime_count"] += 1
        if row.get("labor_hours_estimate") is not None:
            component_bucket["labor_hours_sum"] += float(row["labor_hours_estimate"])
            component_bucket["labor_hours_count"] += 1
        if inspection_age_hours is not None:
            component_bucket["inspection_age_sum"] += inspection_age_hours
            component_bucket["inspection_age_count"] += 1
        if row["event_timestamp"] > component_bucket["last_event_timestamp"]:
            component_bucket["last_event_timestamp"] = row["event_timestamp"]

        tail_number = row["tail_number"]
        current_fleet = fleet_status_acc.get(tail_number)
        if current_fleet is None or row["event_timestamp"] > current_fleet["latest_event_timestamp"] or (
            row["event_timestamp"] == current_fleet["latest_event_timestamp"] and (row.get("sequence_number") or -1) > current_fleet.get("sequence_number", -1)
        ):
            fleet_status_acc[tail_number] = {
                "snapshot_date": now_utc.date().isoformat(),
                "tail_number": tail_number,
                "aircraft_model": row.get("aircraft_model"),
                "airport_code": airport_code,
                "hangar": hangar,
                "latest_event_timestamp": row["event_timestamp"],
                "current_status": row.get("status"),
                "current_severity": row.get("severity"),
                "current_component": component_name,
                "current_maintenance_type": row.get("maintenance_type"),
                "current_fault_code": row.get("fault_code"),
                "current_technician_id": row.get("technician_id"),
                "current_priority": row.get("priority"),
                "current_dispatch_impact": row.get("dispatch_impact"),
                "current_operator_name": row.get("operator_name"),
                "current_route_segment": row.get("route_segment"),
                "current_maintenance_station": row.get("maintenance_station"),
                "current_part_order_status": row.get("part_order_status"),
                "current_estimated_downtime_hours": row.get("estimated_downtime_hours"),
                "current_labor_hours_estimate": row.get("labor_hours_estimate"),
                "open_issue_flag": is_open,
                "aog_flag": is_aog
                ,"requires_parts_flag": requires_parts
                ,"repeat_issue_flag": repeat_issue_flag
                ,"days_since_latest_event": (now_utc.date() - row["event_date"]).days
                ,"gold_loaded_at": now_utc.isoformat().replace("+00:00", "Z")
                ,"sequence_number": row.get("sequence_number") or -1
            }

    daily_operations_preview = []
    for bucket in sorted(daily_operations_acc.values(), key=lambda value: (value["event_date"], value["airport_code"], value["hangar"])):
        daily_operations_preview.append({
            "event_date": bucket["event_date"],
            "airport_code": bucket["airport_code"],
            "hangar": bucket["hangar"],
            "total_events": bucket["total_events"],
            "affected_aircraft": len(bucket["tail_numbers"]),
            "distinct_components": len(bucket["components"]),
            "open_events": bucket["open_events"],
            "aog_events": bucket["aog_events"],
            "critical_events": bucket["critical_events"],
            "high_or_critical_events": bucket["high_or_critical_events"],
            "parts_required_events": bucket["parts_required_events"],
            "repeat_issue_events": bucket["repeat_issue_events"],
            "avg_part_hours": round(bucket["part_hours_sum"] / bucket["part_hours_count"], 2) if bucket["part_hours_count"] else None,
            "avg_estimated_downtime_hours": round(bucket["estimated_downtime_sum"] / bucket["estimated_downtime_count"], 2) if bucket["estimated_downtime_count"] else None,
            "avg_labor_hours_estimate": round(bucket["labor_hours_sum"] / bucket["labor_hours_count"], 2) if bucket["labor_hours_count"] else None,
            "avg_inspection_age_hours": round(bucket["inspection_age_sum"] / bucket["inspection_age_count"], 2) if bucket["inspection_age_count"] else None,
            "last_event_timestamp": bucket["last_event_timestamp"].isoformat().replace("+00:00", "Z"),
            "gold_loaded_at": now_utc.isoformat().replace("+00:00", "Z"),
        })

    component_reliability_preview = []
    for bucket in sorted(component_reliability_acc.values(), key=lambda value: (value["event_date"], value["component"])):
        component_reliability_preview.append({
            "event_date": bucket["event_date"],
            "component": bucket["component"],
            "total_events": bucket["total_events"],
            "affected_aircraft": len(bucket["tail_numbers"]),
            "aog_events": bucket["aog_events"],
            "critical_events": bucket["critical_events"],
            "unscheduled_events": bucket["unscheduled_events"],
            "parts_required_events": bucket["parts_required_events"],
            "repeat_issue_events": bucket["repeat_issue_events"],
            "avg_part_hours": round(bucket["part_hours_sum"] / bucket["part_hours_count"], 2) if bucket["part_hours_count"] else None,
            "avg_estimated_downtime_hours": round(bucket["estimated_downtime_sum"] / bucket["estimated_downtime_count"], 2) if bucket["estimated_downtime_count"] else None,
            "avg_labor_hours_estimate": round(bucket["labor_hours_sum"] / bucket["labor_hours_count"], 2) if bucket["labor_hours_count"] else None,
            "avg_inspection_age_hours": round(bucket["inspection_age_sum"] / bucket["inspection_age_count"], 2) if bucket["inspection_age_count"] else None,
            "last_event_timestamp": bucket["last_event_timestamp"].isoformat().replace("+00:00", "Z"),
            "gold_loaded_at": now_utc.isoformat().replace("+00:00", "Z"),
        })

    fleet_status_preview = []
    for bucket in sorted(fleet_status_acc.values(), key=lambda value: value["tail_number"]):
        fleet_status_preview.append({
            key: value.isoformat().replace("+00:00", "Z") if isinstance(value, datetime) else value
            for key, value in bucket.items()
            if key != "sequence_number"
        })

    print(f"Latest raw file: {latest_file}")
    print(f"Previewing {len(daily_operations_preview[:10])} daily operations rows.")
    (
        daily_operations_preview[:10],
        component_reliability_preview[:10],
        fleet_status_preview[:10],
    )

Latest raw file: jetops-maintenance/evh-herbalife-dev/jetops-maintenance-events-dev/1/2026/04/05/03/49/50.avro
Previewing 10 daily operations rows.


In [ ]:
api_snapshot_root = os.getenv("JETOPS_GOLD_API_SNAPSHOT_ROOT", "jetops/maintenance_kpis_api")
api_snapshot_base_path = f"wasbs://{gold_container}@{storage_account}.blob.core.windows.net/{api_snapshot_root}"

if is_databricks:
    from pyspark.sql.functions import current_timestamp, lit

    def write_api_snapshot(dataframe, relative_path):
        snapshot_path = f"{api_snapshot_base_path}/{relative_path}"
        (
            dataframe.coalesce(1)
            .write
            .mode("overwrite")
            .json(snapshot_path)
        )
        print(f"Wrote API snapshot to {snapshot_path}")

    executive_snapshot_df = fleet_status_df.agg(
        spark_sum("aog_flag").alias("aircraft_in_aog"),
        spark_sum("open_issue_flag").alias("aircraft_with_open_issues"),
        spark_sum("requires_parts_flag").alias("aircraft_requiring_parts"),
        spark_sum("repeat_issue_flag").alias("aircraft_with_repeat_issues"),
        spark_sum(when(col("current_severity") == lit("Critical"), lit(1)).otherwise(lit(0))).alias("aircraft_with_critical_issues"),
        avg("current_estimated_downtime_hours").alias("avg_estimated_downtime_hours"),
        count(lit(1)).alias("fleet_aircraft"),
    ).withColumn("generated_at_utc", current_timestamp())

    metadata_snapshot_df = (
        spark.range(1)
        .select(
            current_timestamp().alias("generated_at_utc"),
            lit(storage_account).alias("storage_account"),
            lit(gold_container).alias("gold_container"),
            lit(api_snapshot_root).alias("api_snapshot_root"),
            lit(daily_operations_dataset).alias("daily_operations_dataset"),
            lit(component_reliability_dataset).alias("component_reliability_dataset"),
            lit(fleet_status_dataset).alias("fleet_status_dataset"),
        )
    )

    write_api_snapshot(executive_snapshot_df, "executive_kpis")
    write_api_snapshot(metadata_snapshot_df, "metadata")
    write_api_snapshot(daily_operations_df, "daily_operations")
    write_api_snapshot(component_reliability_df, "component_reliability")
    write_api_snapshot(fleet_status_df, "fleet_status_snapshot")
else:
    print("Local mode skips API snapshot writes because snapshots are produced from Databricks Gold outputs.")

In [ ]:
if is_databricks:
    print("Daily operations mart preview")
    display(spark.read.format("delta").load(daily_operations_path).orderBy(col("event_date").desc(), col("airport_code"), col("hangar")))

    print("Component reliability mart preview")
    display(spark.read.format("delta").load(component_reliability_path).orderBy(col("event_date").desc(), col("component")))

    print("Fleet status snapshot mart preview")
    display(spark.read.format("delta").load(fleet_status_path).orderBy(col("tail_number")))
else:
    print("Local mode does not write Delta. Use the tuple output from Cell 3 to inspect daily operations, component reliability, and fleet status previews.")